# 07 — Explainability & Interpretability

A forecast that nobody trusts isn't acted on, and a forecast that nobody can
question is even worse. Business stakeholders almost always want to know
**why** the model said what it said:

- "Why are next week's sales higher than last week's?"
- "What if the price of this SKU dropped by 5% — what would the model
  predict?"
- "Did the model use anything weird that it shouldn't have?"

We have three families of techniques to answer these questions:

| Method | Scope | Strength | Weakness |
|---|---|---|---|
| Tree-native importance (`gain`, `split`) | Global | Free, always works | No sign / direction |
| **SHAP** values | Global + Local | Theoretically grounded, signed, additive | More expensive to compute |
| Prophet decomposition | Per-series | Trivially interpretable | Only for Prophet |

In this notebook we will:
1. Train a single LightGBM model and inspect gain / split importance.
2. Compute **global** SHAP importance to see direction of effects.
3. Compute **local** SHAP (a waterfall) to explain a single prediction.
4. Use SHAP **dependence plots** to study individual feature effects.
5. Re-visit Prophet's decomposition as the original interpretable model.


In [3]:
#!pip install shap

In [4]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [5]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
#DATA_PATH    = "/content/sales.csv"             # path to your Kaggle CSV / parquet
#DATE_COL     = "date"                           # date column
#TARGET_COL   = "sales"                          # target / forecast column
#KEY_COLS     = ["store", "item"]                # columns identifying a unique series
#EXCLUDE_COLS = []                               # columns to exlcude
#FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
#HOLDOUT_DAYS = 28                               # length of test horizon

DATA_PATH    = "./dataset/m5/m5_tiny.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
EXCLUDE_COLS = []
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon

## 1. Load data, build features, fit a LightGBM model

In [6]:
from utils.data_utils import load_forecasting_data, complete_panel, time_based_split
from utils.feature_engineering import FeatureEngineer
from utils.lgbm_forecaster import RecursiveForecaster, DEFAULT_PARAMS

raw = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS)
panel = complete_panel(raw, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, fill_value=0.0)
cutoff = panel[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(panel, DATE_COL, cutoff=cutoff)

fe = FeatureEngineer(date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
                     lags=[1, 7, 14, 28], rolling_windows=[7, 28], ewm_halflives=[7.0, 28.0])

forecaster = RecursiveForecaster(feature_engineer=fe, params=DEFAULT_PARAMS,
                            num_boost_round=1500, early_stopping_rounds=100)
forecaster.fit(train_df, valid_df=test_df)
print("forecaster fitted with", len(forecaster.feature_names), "features")


forecaster fitted with 75 features


## 2. Built-in tree importance: `gain` vs `split`

LightGBM tracks two importance measures for free:

- **`split`** = number of times a feature is used to split a node. Quick proxy
  for "how often the model touched this feature".
- **`gain`** = total reduction in loss attributable to that feature. The right
  default — features can have many splits but contribute very little
  improvement.

These tell you *which* features matter, but **not the direction** of their
effect — `gain` is unsigned. For direction we need SHAP (next section).


In [7]:
from utils.explainer import LGBMExplainer
from utils.viz import plot_importance

ex = LGBMExplainer(forecaster.model, feature_names=forecaster.feature_names)
imp_gain  = ex.importance(kind="gain")
imp_split = ex.importance(kind="split")

print("Top 10 by GAIN:")
print(imp_gain.head(10).to_string(index=False))

fig = plot_importance(imp_gain, top_n=20, title="LightGBM importance (gain)")
fig.show()

fig = plot_importance(imp_split, top_n=20, title="LightGBM importance (split count)")
fig.show()


Top 10 by GAIN:
           feature   importance  importance_pct
   sales_ewm_hl7.0 2.326782e+06       47.348159
 sales_roll_mean_7 1.533657e+06       31.208701
           item_id 6.024496e+05       12.259367
sales_roll_mean_28 1.880692e+05        3.827057
       sales_lag_1 6.889742e+04        1.402007
 sales_roll_std_28 3.949638e+04        0.803720
  sales_ewm_hl28.0 3.182805e+04        0.647675
  sales_roll_std_7 2.876620e+04        0.585369
       day_of_week 1.751762e+04        0.356470
      sales_lag_28 1.080908e+04        0.219956


### Reading the chart
Lag-1 and lag-7 features almost always dominate retail forecasting models —
yesterday's sales and same-day-last-week sales are the strongest signals. If
that's not what you see, double-check your feature engineering for leakage or
NaNs.


## 3. SHAP — global view with direction

SHAP (SHapley Additive exPlanations) gives every feature, in every prediction,
a contribution value satisfying the Shapley fairness axioms (efficiency,
symmetry, dummy, additivity). Its key property is **additivity**:

$$ \hat y(x) = \phi_0 + \sum_{j=1}^{p} \phi_j(x) $$

The base value φ₀ plus the per-feature SHAP values exactly equals the
prediction. This makes SHAP simultaneously a **global** and **local**
explanation tool.

We compute SHAP on a sample of validation rows (full computation can be
expensive on large datasets).


In [8]:
import numpy as np
import pandas as pd

# Build a sample to explain (use the test set design matrix)
test_feats = fe.transform(test_df).dropna(subset=[TARGET_COL])
X_test = test_feats[forecaster.feature_names]
sample = X_test.sample(n=min(2000, len(X_test)), random_state=0)
print("explaining", len(sample), "rows")

shap_summary = ex.shap_summary(sample)
shap_summary.head(15)


explaining 2000 rows


,feature,mean_abs_shap,mean_shap
0,sales_ewm_hl7.0,0.950043,-0.950043
1,item_id,0.342449,0.248616
2,sales_roll_mean_7,0.312300,-0.312300
3,sales_roll_std_7,0.252315,-0.252315
4,sales_roll_std_28,0.234101,-0.234101
5,sales_roll_mean_28,0.101298,-0.099752
6,sales_lag_1,0.067206,-0.054807
7,year,0.039357,0.039357
8,sales_ewm_hl28.0,0.034863,0.034605
9,day_of_week,0.026150,0.000483


In [9]:
from utils.viz import plot_shap_summary
fig = plot_shap_summary(shap_summary, top_n=20)
fig.show()


### Interpreting the chart
- **Bar length** = mean(|SHAP|), the average magnitude of the feature's
  contribution. This is the global importance ranking.
- **Color** = signed mean SHAP. Red = pushes predictions *up* on average,
  blue = pushes them *down*. Compare against gain importance: a feature can
  be globally important but with opposing effects in different rows, in which
  case its mean SHAP will be near zero (pale color).


## 4. SHAP local — explaining a single prediction

Pick any row (say the largest forecast in the test set) and decompose its
prediction into per-feature contributions. The waterfall starts at the base
value (the average prediction across the training data), and each bar adds or
subtracts the feature's contribution until we reach the final prediction.


In [10]:
# Find the row with the highest predicted sales
yhat_test = forecaster.model.predict(sample)
top_row_idx = int(np.argmax(yhat_test))
print("max yhat:", yhat_test[top_row_idx])

local = ex.shap_local(sample, row_index=top_row_idx)
print("base value :", local.attrs["base_value"])
print("prediction :", local.attrs["prediction"])
local.head(12)


max yhat: 1.2756262106477207
base value : -0.6811856858966575
prediction : 0.24343720364391597


,feature,value,shap
0,item_id,HOUSEHOLD_1_114,1.404810
1,sales_roll_mean_7,0.0,-0.291805
2,sales_roll_std_7,0.0,-0.223261
3,sales_ewm_hl7.0,0.0,-0.151808
4,sell_price,0.97,0.107221
5,sales_roll_std_28,0.0,-0.083955
6,day_of_week,5,0.073569
7,sales_lag_1,0.0,-0.057859
8,sales_ewm_hl28.0,0.0,0.048119
9,year,2016,0.043537


In [11]:
from utils.viz import plot_shap_waterfall
fig = plot_shap_waterfall(local, top_n=15,
                           title="Why is this prediction high?")
fig.show()


## 5. SHAP dependence plots

For a single feature, plot every row's value vs. its SHAP contribution. This
is the SHAP analogue of a partial-dependence plot but **with interactions
preserved** — vertical spread at a given x-value reveals interactions with
other features.


In [12]:
from utils.viz import plot_shap_dependence

# Pick the most important non-categorical feature
top_feature = next(f for f in shap_summary["feature"] if f not in fe.key_cols)
print("dependence plot for:", top_feature)

dep = ex.shap_dependence(sample, feature=top_feature)
fig = plot_shap_dependence(dep, feature=top_feature)
fig.show()


dependence plot for: sales_ewm_hl7.0


### Reading the dependence plot
- A **monotonically increasing** cloud means: higher feature value → higher
  prediction. Lag features usually look like this.
- A **U-shape** (e.g. for `dayofweek`) means the model has learned a
  cyclical pattern.
- **Vertical dispersion** at a fixed x = interaction effects with other
  features. Heavy dispersion is a signal that the feature alone doesn't tell
  the whole story; investigate interactions with `shap.dependence_plot`'s
  `interaction_index` argument in advanced usage.


## 6. Prophet — interpretable by construction

Prophet doesn't need post-hoc tools because each component is already
human-readable. Let's revisit the decomposition we saw in notebook 06,
this time emphasising the *interpretability angle*.


In [13]:
from utils.prophet_forecaster import MultiKeyProphet
from utils.explainer import prophet_decomposition
from utils.viz import plot_prophet_components

# A small Prophet model on one key, just for the decomposition view.
top_key = (train_df.groupby(KEY_COLS)[TARGET_COL].sum()
           .sort_values(ascending=False).head(1).index[0])
mask = train_df.set_index(KEY_COLS).index == top_key
sub = train_df[mask]

mkp = MultiKeyProphet(date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
                     prophet_kwargs=dict(yearly_seasonality=True, weekly_seasonality=True,
                                         seasonality_mode="multiplicative"))
mkp.fit(sub, verbose=False)
key_tuple = top_key if isinstance(top_key, tuple) else (top_key,)
decomp = prophet_decomposition(mkp.models[key_tuple], key_label=str(top_key))
fig = plot_prophet_components(decomp, title=f"Prophet components — key={top_key}")
fig.show()


05:55:49 - cmdstanpy - INFO - Chain [1] start processing
05:55:49 - cmdstanpy - INFO - Chain [1] done processing


### LightGBM + SHAP vs Prophet decomposition: which to use when?

| Question | Best tool |
|---|---|
| Why is *this specific* forecast high? | LightGBM + SHAP local |
| Which features drive the model overall? | LightGBM + SHAP global |
| What's the trend, and is it accelerating? | Prophet decomposition |
| Did the holiday last week affect demand? | Prophet decomposition (holidays component) |
| What's the marginal effect of price? | LightGBM + SHAP dependence (or Prophet `add_regressor`) |

In practice, run both: LightGBM for accuracy with SHAP for diagnostics,
Prophet alongside it for narrative clarity.


## Recap

- **Gain importance** ranks features but is unsigned.
- **SHAP** assigns signed contributions to every feature in every prediction
  — global ranking and local explanation in one framework.
- **Local waterfall** plots are the single most powerful tool for explaining
  one forecast to a stakeholder.
- **Dependence plots** show shape; vertical spread indicates interactions.
- **Prophet** is interpretable by construction — useful as a complementary
  storytelling tool.

In **notebook 08** we'll bring all four model types — recursive LightGBM,
direct LightGBM, Prophet, and quantile LightGBM — into a single comparison
dashboard with the metrics, charts, and probabilistic visualisations you'd
present to a sponsor.
